In [113]:
import pandas as pd
import numpy as np
from pathlib import Path

In [114]:
# 定义读取文件路径和分表文件存放路径
path = Path(input('输入文件全路径'))
save_path = Path(input('输入结果目录'))

输入文件全路径 D:\Users\Desktop\离职\各中心离职数据明细\2501-2508集团离职明细.xlsx
输入结果目录 D:\Users\Desktop\离职\各中心离职数据明细


In [120]:
df1 = pd.read_excel(path,sheet_name = '离职',header=0,dtype={'工号':str})
df2 = pd.read_excel(path,sheet_name = '202412在职',header=0,dtype={'工号':str})
df3 = pd.read_excel(path,sheet_name = '202508在职',header=0,dtype={'工号':str})
all_df = [df1,df2,df3]
df4 = pd.read_excel(path,sheet_name = 'BP对应表',header=0)
dict1 = dict(zip(df4['事业部'],df4['姓名']))
# df5 = pd.read_excel(path,sheet_name = '看部门离职同比的教程',header = None)

In [121]:
for df in all_df:
    df['离职日期'] = pd.to_datetime(df['离职日期'],format='%Y%m%d',errors = 'coerce')
    df['入职日期'] = pd.to_datetime(df['入职日期'],format='%Y%m%d',errors = 'coerce')

In [122]:
# 应该严格保证df1含的部门严格包含df2和df3
grouped1 = df1.groupby('一级部门')
keys = list(grouped1.groups.keys())
grouped2 = df2.groupby('一级部门')
keys2 = list(grouped2.groups.keys())
grouped3 = df3.groupby('一级部门')
keys3 = list(grouped3.groups.keys())

In [123]:
for key in keys:
    group1 = grouped1.get_group(key).copy()
    group1['工号'] = group1['工号'].astype(str)  # 确保工号是字符串
    if key in keys2:
        group2 = grouped2.get_group(key).copy()
        group2['工号'] = group2['工号'].astype(str)  # 确保工号是字符串
    if key in keys3:
        group3 = grouped3.get_group(key).copy()
        group3['工号'] = group3['工号'].astype(str)  # 确保工号是字符串
    with pd.ExcelWriter(save_path / f'{key}-{dict1.get(key)}.xlsx',engine = 'openpyxl') as writer:
        group1.to_excel(writer,index=False,sheet_name = '2501-2508离职明细')
        worksheet1 = writer.sheets['2501-2508离职明细']
        for row in range(2, worksheet1.max_row + 1):  # 从第2行开始（跳过标题）
            worksheet1.cell(row=row, column=15).number_format = 'yyyy/mm/dd'  # 短日期格式
            worksheet1.cell(row=row, column=26).number_format = 'yyyy/mm/dd'  # 短日期格式
            worksheet1.cell(row=row, column=2).number_format = '@'  # 文本格式
            
        if key in keys2:
            group2.to_excel(writer,index=False,sheet_name = '202412在职')
            worksheet2 = writer.sheets['202412在职']
        for row in range(2, worksheet2.max_row + 1):  # 从第2行开始（跳过标题）
            worksheet2.cell(row=row, column=5).number_format = 'yyyy/mm/dd'  # 短日期格式
            worksheet2.cell(row=row, column=2).number_format = '@'  # 文本格式
            
        if key in keys3:
            group3.to_excel(writer,index=False,sheet_name = '202508在职')
            worksheet3 = writer.sheets['202508在职']
        for row in range(2, worksheet2.max_row + 1):  # 从第2行开始（跳过标题）
            worksheet3.cell(row=row, column=5).number_format = 'yyyy/mm/dd'  # 短日期格式
            worksheet3.cell(row=row, column=2).number_format = '@'  # 文本格式

        # # 教程
        # df5.to_excel(writer,index = False,sheet_name = '看部门离职同比的教程')